In [0]:
from pyspark.sql import functions as F

In [0]:
asset_health = spark.sql("""

SELECT

table_name,

MAX(access_timestamp) as last_access,

DATEDIFF(
CURRENT_DATE(),
DATE(MAX(access_timestamp))
) as days_unused,

SUM(
CASE
WHEN access_type='READ'
THEN 1
ELSE 0
END
) as read_count,

SUM(
CASE
WHEN access_type='WRITE'
THEN 1
ELSE 0
END
) as write_count,

COUNT(*) as total_accesses

FROM platform_monitoring.access_logs

GROUP BY table_name

""")

In [0]:
asset_health.show(
50,
False
)

+------------+--------------------------+-----------+----------+-----------+--------------+
|table_name  |last_access               |days_unused|read_count|write_count|total_accesses|
+------------+--------------------------+-----------+----------+-----------+--------------+
|customers   |2026-06-25 01:00:01.010775|0          |8077      |2022       |10099         |
|products    |2026-06-25 01:00:01.010775|0          |16117     |4036       |20153         |
|payments    |2026-06-25 01:00:01.010775|0          |17268     |4316       |21584         |
|inventory   |2026-06-25 01:00:01.010775|0          |12184     |2940       |15124         |
|orders      |2026-06-25 01:00:01.010775|0          |14336     |3557       |17893         |
|employees   |2026-06-25 01:00:01.010775|0          |3396      |886        |4282          |
|sales       |2026-06-25 01:00:01.010775|0          |7238      |1851       |9089          |
|shipments   |2026-04-26 01:11:15.040927|60         |1141      |289        |1430

In [0]:
asset_health = (

    asset_health

    .withColumn(

        "health_status",

        F.when(
            F.col("days_unused") > 90,
            "CRITICAL"
        )

        .when(
            F.col("days_unused") > 30,
            "WARNING"
        )

        .otherwise(
            "HEALTHY"
        )
    )
)

In [0]:
asset_health.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(
"platform_monitoring.asset_health"
)

In [0]:
spark.table(
"platform_monitoring.asset_health"
).show(
50,
False
)

+------------+--------------------------+-----------+----------+-----------+--------------+-------------+
|table_name  |last_access               |days_unused|read_count|write_count|total_accesses|health_status|
+------------+--------------------------+-----------+----------+-----------+--------------+-------------+
|orders      |2026-06-25 01:00:01.010775|0          |14336     |3557       |17893         |HEALTHY      |
|employees   |2026-06-25 01:00:01.010775|0          |3396      |886        |4282          |HEALTHY      |
|audit_logs  |2026-02-25 01:11:15.040927|120        |37        |11         |48            |CRITICAL     |
|customers   |2026-06-25 01:00:01.010775|0          |8077      |2022       |10099         |HEALTHY      |
|payments    |2026-06-25 01:00:01.010775|0          |17268     |4316       |21584         |HEALTHY      |
|sales       |2026-06-25 01:00:01.010775|0          |7238      |1851       |9089          |HEALTHY      |
|shipments   |2026-04-26 01:11:15.040927|60   

In [0]:
spark.sql("""

SELECT *

FROM platform_monitoring.asset_health

ORDER BY total_accesses DESC

""").show(50,False)

+------------+--------------------------+-----------+----------+-----------+--------------+-------------+
|table_name  |last_access               |days_unused|read_count|write_count|total_accesses|health_status|
+------------+--------------------------+-----------+----------+-----------+--------------+-------------+
|payments    |2026-06-25 01:00:01.010775|0          |17268     |4316       |21584         |HEALTHY      |
|products    |2026-06-25 01:00:01.010775|0          |16117     |4036       |20153         |HEALTHY      |
|orders      |2026-06-25 01:00:01.010775|0          |14336     |3557       |17893         |HEALTHY      |
|inventory   |2026-06-25 01:00:01.010775|0          |12184     |2940       |15124         |HEALTHY      |
|customers   |2026-06-25 01:00:01.010775|0          |8077      |2022       |10099         |HEALTHY      |
|sales       |2026-06-25 01:00:01.010775|0          |7238      |1851       |9089          |HEALTHY      |
|employees   |2026-06-25 01:00:01.010775|0    

In [0]:
%sql
SELECT
table_name,
total_accesses
FROM platform_monitoring.asset_health
ORDER BY total_accesses DESC

table_name,total_accesses
payments,21584
products,20153
orders,17893
inventory,15124
customers,10099
sales,9089
employees,4282
shipments,1430
transactions,298
audit_logs,48


In [0]:
%sql
SELECT
health_status,
COUNT(*)
FROM platform_monitoring.asset_health
GROUP BY health_status

health_status,COUNT(*)
HEALTHY,8
CRITICAL,1
WARNING,1
